We want to check and see if a webserver allows HTTP PUT method for any users.

**We're asking the web server (very nicely :) ) if it accepts HTTP PUT**

In [4]:
# Imports

import requests

Checking the allowed http methods (for us)

In [5]:
target_url = "https://www.irib-news.ir/"

# Send an OPTIONS request to query server configuration
response = requests.options(target_url, timeout=5)

allow_header = response.headers.get("Allow", "")
cors_header = response.headers.get("Access-Control-Allow-Methods", "")

allowed_methods = (allow_header + " " + cors_header).upper()

if "PUT" in allowed_methods:
    print(f"[+] PUT method appears to be allowed by the server configuration.")
    print(f"    Headers found -> Allow: {allow_header} | CORS: {cors_header}")
else:
    print(f"[-] PUT method does not appear to be advertised.")
    print(f"    Server responded with allowed methods: {allow_header if allow_header else 'None listed'}")


[-] PUT method does not appear to be advertised.
    Server responded with allowed methods: None listed


The `Allow` header of the server response claims they do not accept PUT. <br/>
But I'm going to send a HTTP PUT request to it to make sure.

In [6]:
# Test the PUT behavior

probe_filename = "test_connection_probe.txt"
target_endpoint = f"{target_url.rstrip('/')}/{probe_filename}"
payload = "Verification probe: testing HTTP PUT configuration."

# Sending the test PUT request to the target
put_response = requests.put(target_endpoint, data=payload, timeout=5)
status = put_response.status_code

print(f"server's response status code : {status}")

if status in [200, 201, 204]:
    print(f"[!] SUCCESS: The server accepted the PUT request.")
    print(f"    This indicates unauthenticated file creation is ENABLED.")
elif status in [401, 403]:
    print("[-] REFUSED: The server recognizes the PUT method but access was DENIED (Authentication required).")
    print("    The configuration is secure against unauthenticated uploads.")
elif status == 405:
    print("[-] DISABLED: The server explicitly rejected the request with '405 Method Not Allowed'.")
else:
    print(f"[-] INACTIVE: Server returned status {status}. PUT functionality is likely disabled or unmapped.")


server's response status code : 405
[-] DISABLED: The server explicitly rejected the request with '405 Method Not Allowed'.
